# UdaPlay — Part 02: AI Research Agent

Build an agent that:
1. Tries to answer from the local Chroma vector DB (`retrieve_game`).
2. Judges retrieval quality with an LLM (`evaluate_retrieval`).
3. Falls back to Tavily web search when internal knowledge is insufficient (`game_web_search`).
4. Maintains conversation state via the `Agent` state machine.
5. Returns a structured, cited report.

> Run `Udaplay_01_solution_project.ipynb` first so the persistent `chromadb/` folder exists.

### Setup

In [ ]:
import importlib.util, sys
if importlib.util.find_spec('pysqlite3') is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os, json
from typing import List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage, AIMessage, ToolMessage
from lib.tooling import tool

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'Missing OPENAI_API_KEY'
assert os.getenv('TAVILY_API_KEY'), 'Missing TAVILY_API_KEY'
os.environ.setdefault('CHROMA_OPENAI_API_KEY', os.environ['OPENAI_API_KEY'])

MODEL_NAME = 'gpt-4o-mini'

### Re-open the persistent vector store from Part 1

In [ ]:
chroma_client = chromadb.PersistentClient(path='chromadb')
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ['CHROMA_OPENAI_API_KEY'],
    model_name='text-embedding-3-small',
)
collection = chroma_client.get_or_create_collection(
    name='udaplay',
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'},
)
print('Vector DB docs:', collection.count())

tavily_client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])

### Tools

In [ ]:
@tool
def retrieve_game(query: str) -> list:
    """Semantic search over the UdaPlay game vector DB.

    Args:
        query: A natural-language question about a video game (title, platform, year, publisher, genre, etc.).

    Returns:
        A list of up to 5 candidate games. Each item has:
        - id: internal document id
        - distance: cosine distance (lower = closer)
        - Name, Platform, Genre, Publisher, YearOfRelease, Description
    """
    res = collection.query(query_texts=[query], n_results=5)
    out = []
    for i in range(len(res['ids'][0])):
        m = res['metadatas'][0][i]
        out.append({
            'id': res['ids'][0][i],
            'distance': res['distances'][0][i],
            'Name': m.get('Name'),
            'Platform': m.get('Platform'),
            'Genre': m.get('Genre'),
            'Publisher': m.get('Publisher'),
            'YearOfRelease': m.get('YearOfRelease'),
            'Description': m.get('Description'),
        })
    return out

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(..., description='True if the docs sufficiently answer the question.')
    confidence: float = Field(..., ge=0.0, le=1.0, description='0.0 to 1.0 confidence the answer is supported.')
    description: str = Field(..., description='Short justification of the verdict.')


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """LLM-as-judge: decide if retrieved docs can answer the user's question.

    Args:
        question: The original user question.
        retrieved_docs: The list returned by `retrieve_game`.

    Returns a dict with keys: useful (bool), confidence (0..1), description (str).
    Trigger `game_web_search` when useful is False or confidence < 0.6.
    """
    judge = LLM(model=MODEL_NAME, temperature=0.0)
    sys_msg = SystemMessage(content=(
        'You evaluate whether a set of retrieved game records is sufficient to answer a user question. '
        'Be strict: if key facts (year, platform, publisher) are missing or contradictory, mark useful=false. '
        'Respond ONLY as JSON matching the EvaluationReport schema.'
    ))
    user_msg = UserMessage(content=(
        f'Question: {question}\n\n'
        f'Retrieved documents (JSON):\n{json.dumps(retrieved_docs, indent=2)}'
    ))
    ai = judge.invoke([sys_msg, user_msg], response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(ai.content).model_dump()

In [ ]:
@tool
def game_web_search(question: str) -> dict:
    """Search the public web (Tavily) for information about video games.

    Use ONLY when `evaluate_retrieval` reports the local DB is insufficient,
    or when the question concerns very recent / unreleased titles.

    Args:
        question: Natural-language search query.

    Returns a dict with `answer` (Tavily synthesized answer) and `sources` (url + snippet).
    """
    res = tavily_client.search(
        query=question,
        search_depth='advanced',
        include_answer=True,
        max_results=5,
    )
    sources = [
        {'title': r.get('title'), 'url': r.get('url'), 'snippet': (r.get('content') or '')[:400]}
        for r in res.get('results', [])
    ]
    return {'answer': res.get('answer'), 'sources': sources}

### Agent

In [ ]:
INSTRUCTIONS = (
    'You are UdaPlay, a video-game research assistant.\n'
    'Workflow for every user question:\n'
    '  1. Call `retrieve_game` with a focused query.\n'
    '  2. Call `evaluate_retrieval` with the original question and the retrieved docs.\n'
    '  3. If the evaluation says useful=false OR confidence < 0.6, call `game_web_search`.\n'
    '     Otherwise answer directly from the retrieved docs.\n'
    '  4. Write a concise, well-structured final answer.\n'
    'Always cite your sources at the end under a "Sources" section:\n'
    '  - For DB hits cite as `UdaPlay DB: <Name> (<Platform>, <Year>)`.\n'
    '  - For web hits cite the URL.\n'
    'If the facts are uncertain, say so explicitly. Never fabricate release dates or platforms.'
)

agent = Agent(
    model_name=MODEL_NAME,
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2,
)

### Helper to print a clean run report

In [ ]:
def report(run):
    final = run.get_final_state()
    msgs = final['messages']
    print('=' * 70)
    print('Q:', final['user_query'])
    print('-' * 70)
    for m in msgs:
        role = m.role
        if role == 'assistant':
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f'  [tool-call] {tc.function.name}({tc.function.arguments})')
            if m.content:
                print(f'  [assistant] {m.content}')
        elif role == 'tool':
            preview = m.content if len(m.content) < 400 else m.content[:400] + '...'
            print(f'  [tool:{m.name}] {preview}')
    print('-' * 70)
    print('Total tokens:', final.get('total_tokens'))
    print('=' * 70)

### Example queries

In [ ]:
report(agent.invoke('When were Pokémon Gold and Silver released?'))

In [ ]:
report(agent.invoke('Which one was the first 3D platformer Mario game?'))

In [ ]:
report(agent.invoke('Was Mortal Kombat X released for PlayStation 5?'))

In [ ]:
# A question that should fall back to the web
report(agent.invoke('What is Rockstar Games working on right now?'))

### (Optional) Long-term memory: persist web findings back into the vector DB

When the agent falls back to the web, we can store the question + Tavily answer as a new
document in a separate `udaplay_web_memory` collection so future runs can retrieve it without
another web call.

In [ ]:
import hashlib, time

memory_collection = chroma_client.get_or_create_collection(
    name='udaplay_web_memory',
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'},
)

@tool
def remember_web_finding(question: str, answer: str, sources: list) -> str:
    """Persist a web-search finding into long-term memory for future reuse.

    Call after `game_web_search` returns a useful answer. `sources` is the list of
    {title,url,snippet} dicts from the search tool.
    """
    doc = f'Q: {question}\nA: {answer}\nSources: ' + '; '.join(s.get('url', '') for s in sources)
    doc_id = hashlib.sha1(question.encode('utf-8')).hexdigest()[:16]
    memory_collection.upsert(
        ids=[doc_id],
        documents=[doc],
        metadatas=[{'question': question, 'ts': time.time(), 'sources': json.dumps(sources)[:2000]}],
    )
    return f'stored:{doc_id}'


@tool
def recall_web_memory(query: str) -> list:
    """Search prior web findings stored by `remember_web_finding`."""
    if memory_collection.count() == 0:
        return []
    res = memory_collection.query(query_texts=[query], n_results=3)
    return [
        {'id': res['ids'][0][i], 'distance': res['distances'][0][i], 'document': res['documents'][0][i]}
        for i in range(len(res['ids'][0]))
    ]


INSTRUCTIONS_V2 = INSTRUCTIONS + (
    '\nAdditional tools:\n'
    '  - Before calling `game_web_search`, call `recall_web_memory` to check if this was answered before.\n'
    '  - After a successful `game_web_search`, call `remember_web_finding` to persist the finding.\n'
)

agent_v2 = Agent(
    model_name=MODEL_NAME,
    instructions=INSTRUCTIONS_V2,
    tools=[retrieve_game, evaluate_retrieval, game_web_search, recall_web_memory, remember_web_finding],
    temperature=0.2,
)

report(agent_v2.invoke('What did Rockstar Games announce about GTA VI most recently?'))
# Re-asking should hit recall_web_memory instead of the web.
report(agent_v2.invoke('Summarize the latest GTA VI news.'))

### Notes
- The agent is stateful per `session_id` (default `"default"`); call `agent.reset_session()` to clear.
- All tools are plain Python functions decorated with `@tool`; the LLM picks them via OpenAI tool-calling.
- Structured output is enforced for the evaluator via the `EvaluationReport` Pydantic schema.